# Groww Stocks – Data Cleaning
Input: `groww_raw_stocks_industry.csv`. Output: `groww_cleaned_stocks.csv` and `Groww_Cleaned_Data.xlsx`.

Run the cells top to bottom, in the same folder as the input CSV.

## 0. Imports and the cleaning log

In [ ]:
import re
import html
import numpy as np
import pandas as pd

RAW_FILE = "groww_raw_stocks_industry.csv"
log = []          # every cleaning step is recorded here -> becomes the Cleaning Log sheet


def add_log(step, issue, rows, action, why):
    log.append({"Step": step, "Issue found": issue, "Rows affected": int(rows),
                "Action taken": action, "Why": why})

## 1. Load raw data
Read the scraped file (scraper output + industry names). Every cleaning step is also written into a `log` table, which becomes the **Cleaning Log** sheet.

In [ ]:
df = pd.read_csv(RAW_FILE)
n_raw = len(df)
print("Raw shape:", df.shape)

## 2. Text clean-up and duplicates
Remove hidden characters and duplicates so each stock appears once.

In [ ]:
df["Company"] = (df["Company"].astype(str).apply(html.unescape)
                 .str.replace("\xa0", " ", regex=False)
                 .str.replace(r"\s+", " ", regex=True).str.strip())
add_log(1, "Company names had hidden characters (e.g. non-breaking space) and extra spaces",
        (df["Company"] != pd.read_csv(RAW_FILE)["Company"]).sum(),
        "Decoded HTML entities, replaced non-breaking spaces, trimmed spaces",
        "Keeps names consistent for grouping and display")

dups = df.duplicated(subset="URL").sum()
df = df.drop_duplicates(subset="URL")
add_log(2, "Duplicate stock pages (same URL)", dups, "Removed duplicates (none found)",
        "Each stock must appear once")

## 3. Fix the scraper artefact in 52-week low/high
**Scraper artefact.** For stocks with no 52-week data, the scraper grabbed the `52` from the next label (low = high = 52.0). These are set to blank.

In [ ]:
bad52 = (df["Week52_Low"] == 52) & (df["Week52_High"] == 52)
df.loc[bad52, ["Week52_Low", "Week52_High"]] = np.nan
add_log(3, "52-week low AND high were both exactly 52.0 (impossible; scraper picked up the '52' "
           "from the next label when the page had no value)", bad52.sum(),
        "Set both to blank (NaN)", "These stocks have no real 52-week data; keeping 52.0 would be wrong")

## 4. Classify instrument type and trading status
Separate **ETFs** (funds, no company fundamentals) from company **Equity**, and mark **Inactive** stocks (price 0 = suspended, delisted or merged).

In [ ]:
etf_pat = r"\bETF\b|\bBeES\b|Mutual Fund|Index Fund"
df["Instrument_Type"] = np.where(df["Company"].str.contains(etf_pat, case=False, regex=True), "ETF", "Equity")
add_log(4, "ETFs (exchange-traded funds) mixed in with company shares", (df["Instrument_Type"] == "ETF").sum(),
        "Added Instrument_Type = ETF / Equity", "ETFs have no company fundamentals (P/E, ROE...) so they must be analysed separately")

zero_price = df["Price"] == 0
df["Trading_Status"] = np.where(zero_price, "Inactive", "Active")
df.loc[zero_price, "Price"] = np.nan
add_log(5, "Price shown as 0 (suspended, delisted or merged stocks)", zero_price.sum(),
        "Added Trading_Status = Inactive and set Price to blank", "A price of 0 would drag averages down")

df["Exchange_Listing"] = np.where(df["Symbol"].notna(), "NSE listed", "No NSE symbol")
add_log(6, "Symbol is blank for many stocks (the page shows only an 'NSE symbol')", df["Symbol"].isna().sum(),
        "Added Exchange_Listing = NSE listed / No NSE symbol",
        "Assumption: stocks with no NSE symbol are mostly BSE-only listings")

## 5. Missing values
Handle missing values. Financial ratios are **not** imputed, because averaging would invent false numbers. Blank shareholding categories are set to 0 after checking each row then adds up to 100%.

In [ ]:
for c in ["Day_Low", "Day_High"]:
    z = df[c] == 0
    df.loc[z, c] = np.nan
add_log(7, "Day low/high of 0 on a stock with a price", ((df["Price"] > 0) & df["Day_Low"].isna()).sum(),
        "Set zeros to blank", "Zero is not a real trading price")

has_sh = df["Public_pct"].notna()
filled = df.loc[has_sh, ["DII_pct", "FII_pct", "Promoters_pct"]].isna().sum().sum()
df.loc[has_sh, ["DII_pct", "FII_pct", "Promoters_pct"]] = df.loc[has_sh, ["DII_pct", "FII_pct", "Promoters_pct"]].fillna(0)
add_log(8, "DII / FII / Promoter % blank on pages that DO show a shareholding pattern", filled,
        "Filled with 0", "Verified: after filling, DII+FII+Public+Promoters adds up to 100% for every row, "
                         "so a blank means 'no holding'")

df["CEO"] = df["CEO"].fillna("Unknown")
add_log(9, "CEO name missing", (df["CEO"] == "Unknown").sum(), "Filled with 'Unknown'", "Categorical text; avoids dropping rows")

pe_bad = df["PE_Ratio"] <= 0
df.loc[pe_bad, "PE_Ratio"] = np.nan
add_log(10, "P/E ratio of 0 (loss-making or no earnings)", pe_bad.sum(), "Set to blank",
        "P/E is not meaningful when earnings are zero or negative")

df["Founded_Year"] = df["Founded_Year"].astype("Int64")

df = df.drop(columns=["Industry_Method"], errors="ignore")
etf = df["Instrument_Type"] == "ETF"
df["ETF_Tracks"] = np.where(etf, df["Industry"], np.nan)      # index / metal the ETF follows
df.loc[etf, "Industry"] = "ETF (fund)"
add_log(11, "Industry was empty in the first scrape; recovered from the page's hidden JSON data. "
            "For ETFs the 'industry' was really an index name (NIFTY50, SENSEX 50, Gold...)",
        df["Industry"].notna().sum(),
        "Kept Industry for companies; for ETFs moved the index name to ETF_Tracks and set Industry = 'ETF (fund)'",
        "Keeps real industries clean for sector analysis")

fund_cols = ["Market_Cap_Cr", "EPS", "Book_Value"]
df["Has_Fundamentals"] = df[fund_cols].notna().all(axis=1)
add_log(12, "Financial ratios (market cap, EPS, book value, P/E...) missing", (~df["Has_Fundamentals"]).sum(),
        "Left blank, NOT imputed (flagged with Has_Fundamentals)",
        "Averaging or guessing financial ratios would give false numbers")

## 6. Consistency checks and outlier flags (values kept, rows flagged)
Flag odd rows instead of deleting them: price outside the day's range, and extreme ROE / P/E / dividend yield.

In [ ]:
df["Flag_Price_Outside_Day_Range"] = ((df["Price"] < df["Day_Low"] - 1e-9) | (df["Price"] > df["Day_High"] + 1e-9))
add_log(13, "Price outside that day's low-high range", df["Flag_Price_Outside_Day_Range"].sum(),
        "Flagged (kept)", "Likely live-price timing differences, not errors worth deleting")

df["Flag_Outlier"] = ((df["ROE_pct"].abs() > 200) | (df["PE_Ratio"] > 500) | (df["Dividend_Yield_pct"] > 25))
add_log(14, "Extreme values (ROE beyond +/-200%, P/E above 500, dividend yield above 25%)", df["Flag_Outlier"].sum(),
        "Flagged (kept)", "Real but extreme; analysis will use medians or exclude flagged rows")

## 7. Analysis-ready flag
Mark which rows are suitable for company analysis: Equity + Active + has fundamentals.

In [ ]:
df["Analysis_Ready"] = ((df["Instrument_Type"] == "Equity") & (df["Trading_Status"] == "Active") & df["Has_Fundamentals"])
add_log(15, "Rows suitable for company analysis (Equity + Active + has fundamentals)", df["Analysis_Ready"].sum(),
        "Added Analysis_Ready flag", "ETFs, suspended stocks and stocks without fundamentals are kept in the file but excluded from company analysis")

## 8. New columns (feature engineering)
Create new analysis columns (daily change, distance from 52-week high, market-cap bands, company age, institutional holding, profitable, pays dividend).

In [ ]:
df["Day_Change_pct"] = ((df["Price"] - df["Prev_Close"]) / df["Prev_Close"] * 100).round(2)
df["Pct_From_52W_High"] = ((df["Price"] - df["Week52_High"]) / df["Week52_High"] * 100).round(2)
df["Market_Cap_Category"] = pd.cut(df["Market_Cap_Cr"], [0, 500, 5000, 20000, np.inf],
                                   labels=["Micro", "Small", "Mid", "Large"])
df["Company_Age"] = (2026 - df["Founded_Year"]).astype("Int64")
df["Institutional_Holding_pct"] = (df["DII_pct"] + df["FII_pct"]).round(2)
df["Is_Profitable"] = np.where(df["EPS"].isna(), np.nan, (df["EPS"] > 0).astype(float))
df["Pays_Dividend"] = np.where(df["Dividend_Yield_pct"].isna(), np.nan, (df["Dividend_Yield_pct"] > 0).astype(float))
add_log(16, "New analysis columns needed", len(df),
        "Created Day_Change_pct, Pct_From_52W_High, Market_Cap_Category, Company_Age, Institutional_Holding_pct, "
        "Is_Profitable, Pays_Dividend",
        "Market cap bands are approximate rupee thresholds, not the official SEBI ranking")

## 9. Final tidy and save
Order the columns and save the cleaned CSV.

In [ ]:
first = ["Company", "Symbol", "Instrument_Type", "Industry", "Trading_Status", "Exchange_Listing", "Analysis_Ready"]
df = df[first + [c for c in df.columns if c not in first]].reset_index(drop=True)
df.to_csv("groww_cleaned_stocks.csv", index=False)
cleaning_log = pd.DataFrame(log)
print("Cleaned shape:", df.shape, "| analysis-ready rows:", int(df["Analysis_Ready"].sum()))

## 10. Data dictionary
Build the **data dictionary**.

In [ ]:
dd_rows = [
 ("Company", "Text", "Scraped", "Company / instrument name as shown on Groww"),
 ("Symbol", "Text", "Scraped", "NSE trading symbol (blank when the page shows no NSE symbol)"),
 ("Instrument_Type", "Category", "Derived", "Equity (company share) or ETF (exchange-traded fund)"),
 ("Industry", "Category", "Scraped", "Industry / sector from Groww (read from hidden page data); 'ETF (fund)' for ETFs"),
 ("ETF_Tracks", "Text", "Scraped", "Index, commodity or theme an ETF follows (ETFs only)"),
 ("Trading_Status", "Category", "Derived", "Active, or Inactive when the price shows 0 (suspended / delisted / merged)"),
 ("Exchange_Listing", "Category", "Derived", "NSE listed, or No NSE symbol (assumed mostly BSE-only)"),
 ("Analysis_Ready", "True/False", "Derived", "True = Equity + Active + has fundamentals; used for company analysis"),
 ("Price", "Number (Rs)", "Scraped", "Last traded price on the scrape date"),
 ("Prev_Close", "Number (Rs)", "Scraped", "Previous day's closing price"),
 ("Open", "Number (Rs)", "Scraped", "Today's opening price"),
 ("Day_Low", "Number (Rs)", "Scraped", "Lowest price today"),
 ("Day_High", "Number (Rs)", "Scraped", "Highest price today"),
 ("Week52_Low", "Number (Rs)", "Scraped", "Lowest price in the last 52 weeks"),
 ("Week52_High", "Number (Rs)", "Scraped", "Highest price in the last 52 weeks"),
 ("Volume", "Number", "Scraped", "Shares traded today (often blank; not used in headline analysis)"),
 ("Lower_Circuit", "Number (Rs)", "Scraped", "Lowest price the stock may trade at today"),
 ("Upper_Circuit", "Number (Rs)", "Scraped", "Highest price the stock may trade at today"),
 ("Market_Cap_Cr", "Number (Rs Crore)", "Scraped", "Market capitalisation = share price x shares outstanding"),
 ("ROE_pct", "Percent", "Scraped", "Return on equity = net profit / shareholders' equity"),
 ("PE_Ratio", "Number", "Scraped", "Price-to-earnings (TTM); blank when earnings are zero or negative"),
 ("EPS", "Number (Rs)", "Scraped", "Earnings per share (TTM); negative = loss"),
 ("PB_Ratio", "Number", "Scraped", "Price-to-book ratio"),
 ("Dividend_Yield_pct", "Percent", "Scraped", "Annual dividend as % of price"),
 ("Industry_PE", "Number", "Scraped", "Average P/E of the company's industry (same value for all stocks in one industry)"),
 ("Book_Value", "Number (Rs)", "Scraped", "Book value per share"),
 ("Debt_to_Equity", "Number", "Scraped", "Total debt / shareholders' equity"),
 ("Face_Value", "Number (Rs)", "Scraped", "Nominal (par) value of one share"),
 ("CEO", "Text", "Scraped", "CEO / Managing Director ('Unknown' if not shown)"),
 ("Founded_Year", "Integer", "Scraped", "Year the company was founded"),
 ("DII_pct", "Percent", "Scraped", "Holding by domestic institutions (mutual funds, insurers...)"),
 ("FII_pct", "Percent", "Scraped", "Holding by foreign institutional investors"),
 ("Public_pct", "Percent", "Scraped", "Holding by the general public"),
 ("Promoters_pct", "Percent", "Scraped", "Holding by promoters / founders"),
 ("URL", "Text", "Scraped", "Groww page the row was scraped from (unique key)"),
 ("Scraped_At", "Date-time", "Scraped", "When the page was downloaded"),
 ("Has_Fundamentals", "True/False", "Derived", "True when market cap, EPS and book value are all present"),
 ("Flag_Price_Outside_Day_Range", "True/False", "Derived", "Price falls outside today's low-high range (timing difference)"),
 ("Flag_Outlier", "True/False", "Derived", "ROE beyond +/-200%, P/E above 500 or dividend yield above 25%"),
 ("Day_Change_pct", "Percent", "Derived", "(Price - Prev_Close) / Prev_Close x 100"),
 ("Pct_From_52W_High", "Percent", "Derived", "(Price - Week52_High) / Week52_High x 100; 0 means at the 52-week high"),
 ("Market_Cap_Category", "Category", "Derived", "Micro (under Rs 500 Cr), Small (500 to 5,000 Cr), Mid (5,000 to 20,000 Cr), Large (over 20,000 Cr); approximate bands, not the official SEBI ranking"),
 ("Company_Age", "Integer (years)", "Derived", "2026 minus Founded_Year"),
 ("Institutional_Holding_pct", "Percent", "Derived", "DII_pct + FII_pct"),
 ("Is_Profitable", "1 / 0", "Derived", "1 when EPS is above 0"),
 ("Pays_Dividend", "1 / 0", "Derived", "1 when dividend yield is above 0"),
]
data_dictionary = pd.DataFrame(dd_rows, columns=["Column", "Data type", "Source", "Description"])
assert set(data_dictionary["Column"]) == set(df.columns), set(df.columns) ^ set(data_dictionary["Column"])

## 11. Save the Excel workbook
Save the Excel workbook: cleaned data, analysis-ready data, data dictionary and cleaning log.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

with pd.ExcelWriter("Groww_Cleaned_Data.xlsx", engine="openpyxl") as xw:
    df.to_excel(xw, sheet_name="Cleaned_Data", index=False)
    df[df["Analysis_Ready"]].to_excel(xw, sheet_name="Analysis_Ready", index=False)
    data_dictionary.to_excel(xw, sheet_name="Data_Dictionary", index=False)
    cleaning_log.to_excel(xw, sheet_name="Cleaning_Log", index=False)
    for ws in xw.book.worksheets:
        for c in ws[1]:
            c.font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
            c.fill = PatternFill("solid", fgColor="1F3A5F")
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font = Font(name="Arial", size=10)
        ws.freeze_panes = "B2" if ws.title in ("Cleaned_Data", "Analysis_Ready") else "A2"
        for i, col in enumerate(ws.columns, start=1):
            width = max(len(str(c.value)) if c.value is not None else 0 for c in list(col)[:200])
            ws.column_dimensions[get_column_letter(i)].width = min(max(width + 2, 10), 60)
        if ws.title in ("Cleaned_Data", "Analysis_Ready"):
            ws.auto_filter.ref = ws.dimensions
        if ws.title == "Cleaning_Log":
            for row in ws.iter_rows(min_row=2):
                for c in row:
                    c.alignment = Alignment(wrap_text=True, vertical="top")
            for col, w in zip("ABCDE", (6, 60, 14, 55, 65)):
                ws.column_dimensions[col].width = w
print("Saved: groww_cleaned_stocks.csv and Groww_Cleaned_Data.xlsx")

## Check the result

In [ ]:
cleaning_log[["Step","Issue found","Rows affected","Action taken"]]

In [ ]:
df["Analysis_Ready"].value_counts()

In [ ]:
df.head()